# 応用編4
このノートブックでは、共通コードをライブラリ化する例を示します。  
ここでは、サブコントラクトを利用する方法を用います。

## 概要
同一ドメインのサブコントラクト内で、globalオブジェクトに共通的に利用するファンクションを定義しておくことにより、  
親コントラクトから、これをライブラリのように利用することができます。


## 準備

実行の準備を行います。

In [1]:
var { adminWallet, rpc, deploySmartContract, createObjectF } = await import('../lib/notebook-util.mjs');

キーバリューコントラクトを作成します。

In [2]:
var cid_kvs = await createObjectF('keyvalue', 'adv4_kvs');
console.log(cid_kvs);

c238266739


## ライブラリ用コントラクトのデプロイ

ライブラリ用コントラクトを便宜的にファンクションadv4_libで定義します。  
コントラクトの処理として、キーバリューコントラクトにアクセスするファンクションcommon_func_transferをglobalに定義します。  
(common_func_transferはサンプルとしての処理であり、処理の内容に深い意味はありません）

In [3]:
function adv4_lib(){
    var global = (function(){return this})();
    global.common_func_transfer = function common_func_transfer(from, to){
        var cnt = openContract('adv4_kvs');
        var fval = +cnt.get(from);
        var tval = +cnt.get(to);
        if(fval < 0) throw new Error('fval<0');
        if(tval > 0) throw new Error('tval>0');
        fval--;
        tval++;
        cnt.set(from, fval);
        cnt.set(to, tval);
        return [from, fval, to, tval];
    };
}

ライブラリ用コントラクトをデプロイします。

In [4]:
var cid_lib = await deploySmartContract(adv4_lib);
console.log(cid_lib);

c030897355


## スマートコントラクトのデプロイ

共通コードを利用するスマートコントラクトを2つデプロイします。  
両方のコントラクトから同じファンクションcommon_func_transferを利用します。

In [5]:
var cid1 = await deploySmartContract(function adv4_1(){
    openContract('adv4_lib').call(); // ライブラリの読み込み
    // do something
    return common_func_transfer(getCallerId(), getContractId());
});
console.log(cid1);

c089079414


In [6]:
var cid2 = await deploySmartContract(function adv4_2(){
    openContract('adv4_lib').call(); // ライブラリの読み込み
    // do something
    return common_func_transfer(getContractId(), getCallerId());
});
console.log(cid2);

c063406561


デプロイしたスマートコントラクトがライブラリにアクセスできるように許可します。

In [7]:
await rpc.call(adminWallet, 'c1update', { id: cid_lib, prop: 'add accessible_to', value: [cid1, cid2] });

{
  txno: 701999,
  txid: 'x7yPH6SpXKLgjL2nD2pCnJS75LagxCfNvRzggHjWiUeyt',
  status: 'ok',
  value: null
}

デプロイしたスマートコントラクトがキーバリューにアクセスできるように許可します。  
ポイント：ライブラリ用コントラクトに許可は必要ありません。

In [8]:
await rpc.call(adminWallet, 'c1update', { id: cid_kvs, prop: 'add accessible_to', value: [cid1, cid2] });

{
  txno: 702000,
  txid: 'xY8mVmquPMrMNeKSMnDksrVVAYavt57YSUdvFcAKZhm9FB',
  status: 'ok',
  value: null
}

## スマートコントラクトの実行

cid1を実行します。

In [9]:
var resp = await rpc.call(adminWallet, cid1);
console.log(resp);

{
  txno: 702001,
  txid: 'xQUSY8CF86afer5ntbWbbm3WfZuKbqsTR94CEzq7jWD7PB',
  status: 'ok',
  value: [ 'u58281888', -1, 'c089079414', 1 ]
}


cid2を実行します。

In [10]:
var resp = await rpc.call(adminWallet, cid2);
console.log(resp);

{
  txno: 702007,
  txid: 'x6iycYb37DJz7zUrHwgehZyuUUraBcUEGiU8Ku8n9d3YDB',
  status: 'ok',
  value: [ 'c063406561', -1, 'u58281888', 0 ]
}


## エラーになる場合

ここでcid2を実行するとエラーになります。  
（このエラーはサンプルのために起こしたものであり、深い意味はありません）  
スタックトレースから、共通コードの中で例外がスローされたことがわかります。

In [11]:
var resp = await rpc.call(adminWallet, cid2);
console.log(resp);

{
  txno: 702013,
  txid: 'xj5is49TZtDzrSzJav4YMNAAiqYdcLk4xBegi7AS3ogx5',
  status: 'aborted',
  value: 'Error: fval<0\n' +
    '    at common_func_transfer (c030897355:1:256)\n' +
    '    at c063406561:1:53'
}
